# Embedding PE-Core cho keyframe batch2 (Google Colab)

Notebook tự lo hết: **tải zip keyframe từ Hugging Face → giải nén → embedding PE-Core → lưu mỗi video một file `.npz` → upload lại Hugging Face → dọn**.
Bạn chỉ cần: chọn GPU, đăng nhập Hugging Face, bấm chạy các cell theo thứ tự.

**Trước khi chạy:** menu `Runtime → Change runtime type → A100 GPU`.

| Cell | Việc | Ghi chú |
|---|---|---|
| 1 | Cài thư viện + kiểm tra GPU | ~1 phút |
| 2 | Đăng nhập Hugging Face | dán token có quyền **Write** |
| 3 | Cấu hình | thường không cần sửa |
| 4 | Nạp model PE-Core | `hf-hub:timm/PE-Core-bigG-14-448`, fp16 (tải ~5GB lần đầu) |
| 5 | Chạy toàn bộ | tự bỏ qua video đã có kết quả |

**Nguồn keyframe** (đã upload sẵn): `keyframes_batch2/*.zip` (video tách ở máy local) và `keyframes_S01/*.zip` (video S01 tách trên Colab).
Chạy lại cell 5 bất cứ lúc nào để lấy thêm video mới (ví dụ các zip S01 xuất hiện muộn hơn).

**Kết quả** nằm ở `5clnl/idk` → `embeddings_pecore/<tên video>.npz`, mỗi file gồm:
- `frame_idx`: số thứ tự frame (int32, tăng dần)
- `pe_core`: ma trận `[số keyframe, 1280]`, float32, đã chuẩn hoá L2 (giống hệt cách embed bộ dữ liệu cũ)

id đầy đủ của một frame là `<tên video>:<frame_idx>` (ví dụ `M01_V001:25`), cùng dạng với kho dữ liệu cũ.

**Theo dõi tiến độ:** cell 5 in một dòng mỗi lượt (`xong N/614 video (x% ảnh) … còn ≈ m phút`) và một dòng "nhịp tim" mỗi phút. Nếu quá 5 phút không có gì thay đổi nó tự cảnh báo `⚠ KẸT`. Ảnh được đọc bằng luồng (thread), không dùng tiến trình con nên không bị treo ngầm.

**Bị ngắt giữa chừng?** Chạy lại từ cell 1: video nào đã có `.npz` trên Hugging Face sẽ tự được bỏ qua. Kết quả được upload mỗi ~8.000 ảnh (≈4 phút) nên chỉ mất tối đa vài phút công việc.


In [ ]:
#@title 1. Cài thư viện & kiểm tra GPU
# Ghim đúng phiên bản đang dùng ở máy local để embedding khớp với bộ dữ liệu đã có.
!pip install -q "open_clip_torch==3.3.0" "timm==1.0.28"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU. Vào Runtime > Change runtime type > chọn GPU (A100), rồi chạy lại từ cell 1.")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB VRAM")


In [ ]:
#@title 2. Đăng nhập Hugging Face
# Hiện ô nhập token: dán token có quyền Write (huggingface.co/settings/tokens).
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # 614 zip x 2 thanh tiến trình = hàng nghìn dòng, làm nặng tab Colab
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 3. Cấu hình
import os
REPO = "5clnl/idk"
SOURCE_DIRS = ["keyframes_batch2", "keyframes_S01"]   # thư mục chứa zip keyframe trên HF
OUT_DIR = "embeddings_pecore"                         # thư mục kết quả trên HF

BATCH_SIZE = 64            # số ảnh mỗi lượt qua GPU (A100 40GB dư sức; giảm nếu GPU nhỏ hơn)
NUM_LOADERS = max(2, min(8, os.cpu_count() or 2))    # số LUỒNG CPU đọc + resize ảnh
GROUP_MIN_IMAGES = 4000    # gom nhiều video nhỏ vào một lượt để GPU không phải chờ
UPLOAD_EVERY_IMAGES = 8000   # upload kết quả lên HF sau mỗi ngần này ảnh (~4 phút)
LOAD_TIMEOUT = 300         # một batch ảnh đọc quá lâu (giây) thì báo lỗi thay vì treo
HEARTBEAT_SEC = 60         # in một dòng trạng thái mỗi ngần này giây
print(f"{os.cpu_count()} CPU -> {NUM_LOADERS} luồng đọc ảnh, batch {BATCH_SIZE}")


In [ ]:
#@title 4. Nạp model PE-Core
import os, re, time, json, queue, shutil, threading, zipfile, itertools
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, torch, open_clip
from PIL import Image

EMB_DIM = 1280
device = "cuda"
print("Đang nạp hf-hub:timm/PE-Core-bigG-14-448 (lần đầu tải ~5GB)...")
t0 = time.time()
model, _, preprocess = open_clip.create_model_and_transforms("hf-hub:timm/PE-Core-bigG-14-448", device="cpu")
model = model.half().to(device).eval()
print(f"Model sẵn sàng sau {time.time()-t0:.0f}s, VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")


def _load_batch(batch):
    return torch.stack([preprocess(Image.open(p).convert("RGB")) for p in batch])


@torch.inference_mode()
def embed_paths(paths):
    """Embedding PE-Core (fp16, chuẩn hoá L2) cho danh sách ảnh, giữ nguyên thứ tự.

    Ảnh được đọc + resize bằng THREAD, không dùng DataLoader nhiều tiến trình: fork trong lúc luồng tải zip
    đang chạy có thể treo im lặng (đã gặp ở lượt thứ 3). Đọc quá LOAD_TIMEOUT giây thì báo lỗi."""
    out = np.empty((len(paths), EMB_DIM), np.float32)
    batches = [paths[i:i + BATCH_SIZE] for i in range(0, len(paths), BATCH_SIZE)]
    it = iter(batches)
    ex = ThreadPoolExecutor(max_workers=max(1, NUM_LOADERS))
    k = 0
    try:
        window = deque(ex.submit(_load_batch, b) for b in itertools.islice(it, max(2, NUM_LOADERS + 2)))
        while window:
            xb = window.popleft().result(timeout=LOAD_TIMEOUT)
            nxt = next(it, None)
            if nxt is not None:
                window.append(ex.submit(_load_batch, nxt))
            f = model.encode_image(xb.to(device, non_blocking=True).half())
            f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            assert f.shape[1] == EMB_DIM, f.shape
            out[k:k + len(f)] = f.float().cpu().numpy()
            k += len(f)
    finally:
        ex.shutdown(wait=False, cancel_futures=True)
    assert k == len(paths), (k, len(paths))
    return out


In [ ]:
#@title 5. Chạy: tải zip → embedding → lưu .npz → upload → dọn
import zipfile
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()          # tránh hàng nghìn dòng thanh tiến trình làm nặng tab Colab

api = HfApi()
WORK = Path("/content/work")
OUT = Path("/content/embeddings_out")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)
FRAME_RE = re.compile(r"^(.+?)_frame_(\d+)\.jpg$")


def log(msg):
    print(msg, flush=True)


# ---- liệt kê video có sẵn và video đã có kết quả
files = api.list_repo_files(REPO, repo_type="dataset")
zips = {}
for f in files:
    for d in SOURCE_DIRS:
        if f.startswith(d + "/") and f.endswith(".zip"):
            v = Path(f).stem
            if v in zips:
                raise RuntimeError(f"video {v} xuất hiện ở cả {zips[v]} và {d}")
            zips[v] = f
done = {Path(f).stem for f in files if f.startswith(OUT_DIR + "/") and f.endswith(".npz")}
pending = sorted(v for v in zips if v not in done)
log(f"Có {len(zips)} video trên HF, đã xong {len(done & set(zips))}, còn {len(pending)} video cần embedding")

# ---- số keyframe mong đợi của từng video (để phát hiện zip thiếu/hỏng)
expected = {}
if "keyframes_batch2/index.json" in files:
    idx_path = hf_hub_download(REPO, "keyframes_batch2/index.json", repo_type="dataset", local_dir=str(WORK / "meta"))
    for v, x in json.load(open(idx_path))["videos"].items():
        expected[v] = x["frames"]


def expected_frames(v):
    if v in expected:
        return expected[v]
    jp = f"keyframes_S01/{v}.json"
    if jp in files:
        p = hf_hub_download(REPO, jp, repo_type="dataset", local_dir=str(WORK / "meta"))
        return json.load(open(p))["keyframes"]
    return None


def prepare(v):
    """Tải + giải nén zip của video v, kiểm tra số ảnh. Trả về danh sách (frame_idx, đường dẫn)."""
    zp = hf_hub_download(REPO, zips[v], repo_type="dataset", local_dir=str(WORK / "zips"))
    vdir = WORK / "img" / v
    if vdir.exists():
        shutil.rmtree(vdir)
    vdir.mkdir(parents=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(vdir)
    os.remove(zp)
    items = []
    for p in vdir.glob("*.jpg"):
        m = FRAME_RE.match(p.name)
        if m and m.group(1) == v:
            items.append((int(m.group(2)), str(p)))
    items.sort()
    want = expected_frames(v)
    if want is not None and len(items) != want:
        raise RuntimeError(f"zip có {len(items)} ảnh, đáng lẽ phải có {want}")
    if not items:
        raise RuntimeError("zip không có ảnh nào")
    return items


# ---- để tính % và thời gian còn lại
total_vid = len(pending)
total_img = sum((expected_frames(v) or 196) for v in pending)
log(f"Tổng cần embedding ≈ {total_img:,} ảnh")

S = {"downloaded": 0, "busy": False, "busy_since": 0.0, "done_vid": 0, "done_img": 0, "last_change": time.time()}


def touch():
    S["last_change"] = time.time()


# ---- luồng chuẩn bị: tải + giải nén trước trong khi GPU đang chạy lượt trước
q = queue.Queue(maxsize=2)
failed = []


def producer():
    group, n_img = [], 0
    for v in pending:
        try:
            items = prepare(v)
        except Exception as e:
            log(f"[{v}] !! LỖI khi chuẩn bị: {e}")
            failed.append(v)
            shutil.rmtree(WORK / "img" / v, ignore_errors=True)
            continue
        S["downloaded"] += 1
        touch()
        group.append((v, items))
        n_img += len(items)
        if n_img >= GROUP_MIN_IMAGES:
            q.put(group)
            group, n_img = [], 0
    if group:
        q.put(group)
    q.put(None)


prod = threading.Thread(target=producer, daemon=True)
prod.start()

# ---- nhịp tim: một dòng mỗi HEARTBEAT_SEC giây, tự cảnh báo khi không có gì thay đổi > 5 phút
stop = threading.Event()


def heartbeat():
    while not stop.wait(HEARTBEAT_SEC):
        idle = time.time() - S["last_change"]
        gpu = (f"GPU đang chạy lượt hiện tại {(time.time() - S['busy_since']) / 60:.1f} phút" if S["busy"]
               else "GPU đang chờ dữ liệu")
        warn = f"   ⚠ KẸT? {idle / 60:.0f} phút không có gì thay đổi" if idle > 300 else ""
        log(f"[{time.strftime('%H:%M:%S')}] tải xong {S['downloaded']}/{total_vid} | hàng đợi {q.qsize()}/2 | {gpu} | "
            f"đã lưu {S['done_vid']}/{total_vid} video ({100 * S['done_img'] / max(1, total_img):.1f}%){warn}")


threading.Thread(target=heartbeat, daemon=True).start()


def upload_outputs():
    api.upload_folder(repo_id=REPO, repo_type="dataset", folder_path=str(OUT), path_in_repo=OUT_DIR,
                      allow_patterns=["*.npz"], commit_message="PE-Core embeddings (Colab)")


def try_upload():
    """Upload có thử lại; lỗi mạng thoáng qua không làm sập cả lượt chạy."""
    for attempt in range(1, 4):
        try:
            upload_outputs()
            return True
        except Exception as e:
            log(f"   upload lỗi lần {attempt}/3: {type(e).__name__}: {e}")
            time.sleep(10 * attempt)
    return False


def embed_group(group):
    """[(video, items, embeddings|None)]. Nếu cả lượt lỗi thì thử từng video để cô lập video hỏng."""
    try:
        embs = embed_paths([p for _, items in group for _, p in items])
        out, k = [], 0
        for v, items in group:
            out.append((v, items, embs[k:k + len(items)]))
            k += len(items)
        return out
    except Exception as e:
        log(f"!! lượt {len(group)} video bị lỗi ({type(e).__name__}: {e}) — thử lại từng video")
        out = []
        for v, items in group:
            try:
                out.append((v, items, embed_paths([p for _, p in items])))
            except Exception as e2:
                log(f"[{v}] !! LỖI khi embedding: {type(e2).__name__}: {e2}")
                out.append((v, items, None))
        return out


t_start = time.time()
since_upload = 0
unuploaded = []
try:
    while True:
        try:
            group = q.get(timeout=60)
        except queue.Empty:
            if not prod.is_alive() and q.empty():
                log("!! luồng tải đã dừng bất thường — kết thúc sớm; chạy lại cell này để tiếp tục")
                break
            continue
        if group is None:
            break
        S["busy"], S["busy_since"] = True, time.time()
        touch()
        t0 = time.time()
        n_group_img = sum(len(items) for _, items in group)
        for v, items, e in embed_group(group):
            if e is None:
                failed.append(v)
                continue
            if not np.isfinite(e).all() or abs(np.linalg.norm(e, axis=1) - 1).max() > 1e-2:
                log(f"[{v}] !! embedding không hợp lệ (NaN hoặc chưa chuẩn hoá), bỏ qua")
                failed.append(v)
                continue
            tmp = OUT / f"{v}.tmp.npz"
            np.savez(tmp, frame_idx=np.array([i for i, _ in items], np.int32), pe_core=e)
            os.replace(tmp, OUT / f"{v}.npz")
            unuploaded.append(v)
            S["done_vid"] += 1
        for v, _ in group:
            shutil.rmtree(WORK / "img" / v, ignore_errors=True)
        S["done_img"] += n_group_img
        since_upload += n_group_img
        S["busy"] = False
        touch()
        dt = time.time() - t0
        el = time.time() - t_start
        eta = max(0, total_img - S["done_img"]) / max(S["done_img"] / max(el, 1e-6), 1e-6) / 60
        log(f"lượt {len(group)} video / {n_group_img} ảnh: {n_group_img / dt:.1f} ảnh/s | "
            f"xong {S['done_vid']}/{total_vid} video, {S['done_img']:,}/{total_img:,} ảnh ({100 * S['done_img'] / max(1, total_img):.1f}%) | "
            f"đã chạy {el / 60:.1f} phút, còn ≈ {eta:.0f} phút")
        if since_upload >= UPLOAD_EVERY_IMAGES:
            if try_upload():
                log(f"   đã upload {len(unuploaded)} file kết quả lên {REPO}/{OUT_DIR}/")
                unuploaded, since_upload = [], 0
            else:
                log("   !! upload chưa được, sẽ thử lại ở lượt sau (kết quả vẫn còn trong /content/embeddings_out)")

    if unuploaded:
        if try_upload():
            log(f"   đã upload {len(unuploaded)} file kết quả lên {REPO}/{OUT_DIR}/")
        else:
            log("   !! KHÔNG upload được phần cuối — chạy lại cell này để upload nốt")
finally:
    stop.set()

# ---- kiểm tra cuối: so danh sách kết quả trên HF với video đã có zip
remote_done = {Path(f).stem for f in api.list_repo_files(REPO, repo_type="dataset") if f.startswith(OUT_DIR + "/") and f.endswith(".npz")}
missing = sorted(set(zips) - remote_done)
log("\n================ TỔNG KẾT ================")
log(f"Embedding xong: {S['done_vid']} video / {S['done_img']} ảnh trong {(time.time() - t_start) / 60:.1f} phút")
log(f"Trên HF hiện có {len(remote_done & set(zips))}/{len(zips)} video có kết quả")
if failed:
    log("Lỗi (chạy lại cell này để thử lại): " + ", ".join(sorted(set(failed))))
if missing and not failed:
    log("Chưa có kết quả: " + ", ".join(missing) + "  (có thể zip của chúng vừa mới xuất hiện — chạy lại cell này)")
if not missing and not failed:
    log("Tất cả đã xong.")
